# 02 - EDA: Feature Exploration

Inputs: WKA power, weather forecasts, Q Rothenstein (15min + daily).  
Goals: timezone audit, Q vs power, weather vs power, lag validation, feature summary.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 4)})

## 1 - Load All Sources

In [ ]:
RAW_WKA = '../data/raw/Zeitreihen_WKA_2025_2026.xlsx'

# Rows 0-4 metadata, row 5 = column headers, row 6+ = data
raw = pd.read_excel(RAW_WKA, header=None)
wka = raw.iloc[6:].copy()
wka.columns = ['Datum_von', 'Datum_bis', 'Unterpreilipp_kW', 'Burgau_kW']

# Raw timestamps are UTC+1 fixed offset (no DST — Lokale Zeit: Nein in metadata)
# Subtract 1 h then tag as UTC — avoids DST ambiguity on 2025-10-26
wka['Datum_von'] = (
    (pd.to_datetime(wka['Datum_von'], dayfirst=True) - pd.Timedelta(hours=1))
    .dt.tz_localize('UTC')
)
wka['Unterpreilipp_kW'] = pd.to_numeric(wka['Unterpreilipp_kW'], errors='coerce')
wka['Burgau_kW']        = pd.to_numeric(wka['Burgau_kW'],        errors='coerce')
wka = wka.drop_duplicates(subset='Datum_von').set_index('Datum_von').drop(columns=['Datum_bis']).sort_index()

print('Shape :', wka.shape)
print('Range :', wka.index.min(), '->', wka.index.max())
print('TZ    :', wka.index.tz)
print('Nulls :', wka.isnull().sum().to_dict())
wka.head(3)

In [ ]:
RAW_WX = '../data/raw/Zeitreihen_Wetterprognosen_2025_2026.xlsx'

raw_wx = pd.read_excel(RAW_WX, header=None)
wx = raw_wx.iloc[6:].copy()
wx.columns = ['Datum_von', 'Datum_bis', 'solar_W_m2', 'temp_C', 'wind_ms', 'wind_dir_deg']
wx['Datum_von'] = (
    (pd.to_datetime(wx['Datum_von'], dayfirst=True) - pd.Timedelta(hours=1))
    .dt.tz_localize('UTC')
)
for col in ['solar_W_m2', 'temp_C', 'wind_ms', 'wind_dir_deg']:
    wx[col] = pd.to_numeric(wx[col], errors='coerce')
wx = wx.drop_duplicates(subset='Datum_von').set_index('Datum_von').drop(columns=['Datum_bis']).sort_index()
print('Weather:', wx.index.min(), '->', wx.index.max(), f'({len(wx):,} rows)')

In [ ]:
q15 = pd.read_csv('../data/raw/water_data/Q_Rothenstein_15min.csv')
q15['phenomenonTime'] = pd.to_datetime(q15['phenomenonTime'], utc=True)
q15 = q15.set_index('phenomenonTime').sort_index()
q15.columns = ['Q_m3s']
print('Q 15min:', q15.index.min(), '->', q15.index.max(), f'({len(q15):,} rows)')

q1d = pd.read_csv('../data/raw/water_data/Q_Rothenstein_1D.csv')
q1d['phenomenonTime'] = pd.to_datetime(q1d['phenomenonTime'], utc=True)
q1d = q1d.set_index('phenomenonTime').sort_index()
q1d.columns = ['Q_m3s']
print('Q daily:', q1d.index.min(), '->', q1d.index.max(), f'({len(q1d):,} rows)')

## 2 - Timezone Audit

In [ ]:
for name, df in [('Power', wka), ('Weather', wx), ('Q 15min', q15), ('Q daily', q1d)]:
    print(f'{name:12s}  tz = {df.index.tz}')

## 3 - Discharge Q vs Power

In [ ]:
power_daily = wka[['Unterpreilipp_kW','Burgau_kW']].resample('D').mean()
merged_1d = power_daily.join(q1d.resample('D').mean(), how='inner')
merged_1d = merged_1d[(merged_1d['Unterpreilipp_kW']>0)&(merged_1d['Burgau_kW']>0)]
print('Daily merge:', merged_1d.shape)
for col in ['Unterpreilipp_kW','Burgau_kW']:
    print(f'  r(Q, {col}) = {merged_1d[col].corr(merged_1d["Q_m3s"]):.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, c in zip(axes, ['Unterpreilipp_kW','Burgau_kW'], ['steelblue','darkorange']):
    ax.scatter(merged_1d['Q_m3s'], merged_1d[col], alpha=0.3, s=8, color=c)
    ax.set_xlabel('Q (m3/s)'); ax.set_ylabel('kW')
    ax.set_title(f'{col}  r={merged_1d[col].corr(merged_1d["Q_m3s"]):.3f}')
plt.suptitle('Daily Q vs Power'); plt.tight_layout(); plt.show()

In [ ]:
merged_15 = wka.join(q15, how='inner')
merged_15 = merged_15[merged_15['Unterpreilipp_kW'] > 0]
print('15-min merge:', merged_15.shape)
for col in ['Unterpreilipp_kW','Burgau_kW']:
    print(f'  r(Q, {col}) at 15-min = {merged_15[col].corr(merged_15["Q_m3s"]):.3f}')

## 4 - Weather vs Power

In [ ]:
merged_wx = wka.join(wx, how='inner')
merged_wx = merged_wx[merged_wx['Unterpreilipp_kW'] > 0]
features  = ['solar_W_m2', 'temp_C', 'wind_ms']
print('Pearson r with Unterpreilipp_kW:')
for f in features:
    print(f'  {f:20s}  r = {merged_wx["Unterpreilipp_kW"].corr(merged_wx[f]):.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, feat in zip(axes, features):
    ax.scatter(merged_wx[feat], merged_wx['Unterpreilipp_kW'], alpha=0.15, s=4, color='steelblue')
    ax.set_xlabel(feat); ax.set_ylabel('Unterpreilipp kW')
    ax.set_title(f'r = {merged_wx["Unterpreilipp_kW"].corr(merged_wx[feat]):.3f}')
plt.suptitle('Weather vs power'); plt.tight_layout(); plt.show()

## 5 - Lag Validation

In [ ]:
series = wka['Unterpreilipp_kW'].dropna()
series = series[series > 0]
rows = []
for lag, label in [(1,'15min'),(4,'1h'),(8,'2h'),(96,'1day'),(192,'2days'),(672,'1week')]:
    rows.append({'lag': lag, 'offset': label, 'r': round(series.corr(series.shift(lag)), 3)})
lag_df = pd.DataFrame(rows)
print(lag_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(lag_df['offset'], lag_df['r'], color='steelblue', edgecolor='white')
ax.bar_label(bars, fmt='%.3f', fontsize=9)
ax.set_xlabel('Lag'); ax.set_ylabel('Pearson r'); ax.axhline(0, color='black', lw=0.8)
ax.set_title('Lag correlation - Unterpreilipp')
ax.grid(axis='y', alpha=0.3); plt.tight_layout(); plt.show()

## 6 - Feature Summary

| Feature | Source | Available at forecast time | Signal |
|---|---|---|---|
| lag_1 (t-15min) | Power history | Yes | High |
| lag_4 (t-1h) | Power history | Yes | High |
| lag_96 (t-1day) | Power history | Yes | Highest |
| Q_m3s upstream | Gauge API | Yes (real-time) | High |
| solar_W_m2 | DWD forecast | Yes | Low-medium |
| temp_C | DWD forecast | Yes | Low |
| wind_ms | DWD forecast | Yes | Low |
| hour, month, weekday | Calendar | Always | Medium |
| plant_offline | Derived (Burgau zeros) | Yes | Burgau only |

**Key takeaway:** lag_96 and Q are the two strongest features. Weather adds marginal signal but strengthens the explainability story for the jury.